In [2]:
import sqlite3
import pandas as pd
import logging
from Ingestion_db import ingest_db

logging.basicConfig(filename='logs/vendor_summary.log', level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s', filemode='a')

def create_vendor_summary(conn):
    '''this function will merge the different tables to get the overall vendor summary and adding new columns in the resultant data'''

    vendor_sales_summary = pd.read_sql_query("""WITH FrieghtSummary AS (
    SELECT VendorNumber, SUM(Freight) as FreightCost FROM vendor_invoice GROUP BY VendorNumber),
                                PurchaseSummary AS (
    SELECT p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.price AS Actualprice, pp.Volume, SUM(p.Quantity) as TotalPurchaseQuantity, SUM(p.Dollars) as TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.price, pp.Volume),

    SalesSummary AS (
    SELECT VendorNo, Brand, SUM(SalesDollars) as TotalSalesDollars, SUM(SalesPrice) as TotalSalesPrice, SUM(SalesQuantity) as TotalSalesQuantity, SUM(ExciseTax) as TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand)

    SELECT 
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.Actualprice,
    ps.Volume,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice,
    ss.TotalSalesQuantity,
    ss.TotalExciseTax,
    fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNo 
    AND ps.Brand = ss.Brand
    LEFT JOIN FrieghtSummary fs
    ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC
    """, conn)

    return vendor_sales_summary

def clean_data(df):
    '''this function will clean the data'''
    df['Volume'] = df['Volume'].astype(float)

    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    df.fillna(0, inplace=True)

    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars'].replace(0, pd.NA)) * 100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity'].replace(0, pd.NA)
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars'].replace(0, pd.NA)

    df.fillna(0, inplace=True)

    return df

if __name__ == '__main__':
    # creating database connection
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table.....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data.....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info('Ingesting data.....')
    ingest_db(clean_df, 'vendor_sales_summary', conn)
    logging.info('Completed')

C:\Users\prane\AppData\Local\Temp\ipykernel_428\4227884926.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0, inplace=True)
